In [ ]:

%pip install -q qiskit numpy matplotlib


In [ ]:

import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, state_fidelity

# Four-qubit experimental model
N_QUBITS = 4

def generate_state(key):
    """Create a quantum state from a 6-bit key."""

    if not 0 <= key < 64:
        raise ValueError("Key must be between 0 and 63")

    qc = QuantumCircuit(N_QUBITS)

    # Encode the key into rotation angles
    bits = [(key >> i) & 1 for i in range(6)]

    for q in range(N_QUBITS):
        angle = np.pi / 4 + bits[q] * np.pi / 2
        qc.ry(angle, q)

    # Entangle the qubits
    qc.cx(0, 1)
    qc.cx(1, 2)
    qc.cx(2, 3)

    # Additional key-dependent gates
    qc.rz(bits[4] * np.pi / 2, 1)
    qc.rx(bits[5] * np.pi / 2, 2)

    return qc

# Generate a state using secret key 13
secret_key = 13
circuit = generate_state(secret_key)

print(circuit.draw(output="text"))

secret_state = Statevector.from_instruction(circuit)

# Compare with a different key
candidate_state = Statevector.from_instruction(
    generate_state(25)
)

fidelity = state_fidelity(
    secret_state,
    candidate_state
)

print("State fidelity:", fidelity)


In [ ]:

# Experiment 2: Exhaustive key-recovery benchmark

target_state = Statevector.from_instruction(
    generate_state(secret_key)
)

results = []

for candidate_key in range(64):
    candidate_state = Statevector.from_instruction(
        generate_state(candidate_key)
    )

    fidelity = state_fidelity(
        target_state, candidate_state
    )

    results.append((candidate_key, fidelity))

# Rank candidate keys by state fidelity
results.sort(key=lambda x: x[1], reverse=True)

print("Actual secret key:", secret_key)
print("\nTop 10 candidate keys:")

for key, fidelity in results[:10]:
    print(f"Key {key:2d}: Fidelity = {fidelity:.6f}")

# Check for keys producing the same quantum state
matching_keys = [
    key for key, fidelity in results
    if np.isclose(fidelity, 1.0)
]

print("\nKeys producing equivalent states:", matching_keys)


In [ ]:

from qiskit.quantum_info import Statevector
import numpy as np

secret_key = 13
shots = 100
rng = np.random.default_rng(42)

# Prepare target state
target = Statevector.from_instruction(
    generate_state(secret_key)
)

# Attacker receives only measurement outcomes
target.seed(42)
samples = target.sample_counts(shots=shots)


print("Target measurement counts:", samples)

# Convert measured bitstrings into frequencies
observed = np.zeros(16)

for bitstring, count in samples.items():
    observed[int(bitstring, 2)] = count / shots

# Evaluate all possible keys using the public circuit
scores = []

for candidate_key in range(64):
    candidate = Statevector.from_instruction(
        generate_state(candidate_key)
    )

    probabilities = candidate.probabilities()

    # Squared difference between distributions
    error = np.sum(
        (observed - probabilities) ** 2
    )

    scores.append((candidate_key, error))

scores.sort(key=lambda x: x[1])

print("\nActual secret key:", secret_key)
print("Top 10 candidate keys:")

for key, error in scores[:10]:
    print(f"Key {key:2d}: Error = {error:.6f}")

print("\nPredicted key:", scores[0][0])


In [ ]:

import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

rng = np.random.default_rng(42)

secret_key = 13
shots_per_basis = 100

def probabilities_in_basis(key, basis):
    qc = generate_state(key)

    if basis == "X":
        for q in range(4):
            qc.h(q)

    state = Statevector.from_instruction(qc)
    return state.probabilities()

# Simulate measurements of the secret state
observations = {}

for basis in ["Z", "X"]:
    probabilities = probabilities_in_basis(
        secret_key, basis
    )

    observations[basis] = rng.multinomial(
        shots_per_basis, probabilities
    )

# Attacker tests every possible key
results = []

for candidate_key in range(64):
    log_likelihood = 0.0

    for basis in ["Z", "X"]:
        probabilities = probabilities_in_basis(
            candidate_key, basis
        )

        counts = observations[basis]

        # Penalize impossible predictions
        log_likelihood += np.sum(
            counts * np.log(
                np.maximum(probabilities, 1e-12)
            )
        )

    results.append(
        (candidate_key, log_likelihood)
    )

results.sort(key=lambda x: x[1], reverse=True)

print("Actual secret key:", secret_key)
print("\nTop 10 candidates:")

for key, score in results[:10]:
    print(
        f"Key {key:2d}: "
        f"Log-likelihood = {score:.3f}"
    )

print("\nPredicted key:", results[0][0])


In [ ]:

import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

N_QUBITS = 4
SECRET_KEY = 13
KEYSPACE = 64

def scrambling_circuit(key, depth=4):
    if not 0 <= key < KEYSPACE:
        raise ValueError("Key must be between 0 and 63")

    qc = QuantumCircuit(N_QUBITS)

    # Reuse the same key encoding as our baseline
    bits = [(key >> i) & 1 for i in range(6)]

    for q in range(N_QUBITS):
        angle = np.pi / 4 + bits[q] * np.pi / 2
        qc.ry(angle, q)

    qc.rz(bits[4] * np.pi / 2, 1)
    qc.rx(bits[5] * np.pi / 2, 2)

    # Repeated scrambling-inspired layers
    for layer in range(depth):
        for q in range(N_QUBITS):
            qc.ry(0.37 * (q + 1) * (layer + 1), q)
            qc.rz(0.23 * (q + 1) * (layer + 1), q)

        # Entangling gates
        qc.cx(0, 1)
        qc.cx(2, 3)
        qc.cx(1, 2)
        qc.cx(3, 0)

    return qc

circuit = scrambling_circuit(SECRET_KEY, depth=4)
print(circuit.draw(output="text"))


In [ ]:

from qiskit.quantum_info import Statevector

SHOTS = 100
rng = np.random.default_rng(42)

def basis_probabilities(key, depth, basis):
    qc = scrambling_circuit(key, depth)

    if basis == "X":
        for q in range(N_QUBITS):
            qc.h(q)

    return Statevector.from_instruction(qc).probabilities()

def attack(depth):
    observations = {}

    # Sample the secret state
    for basis in ["Z", "X"]:
        p = basis_probabilities(
            SECRET_KEY, depth, basis
        )
        observations[basis] = rng.multinomial(SHOTS, p)

    scores = []

    for candidate_key in range(KEYSPACE):
        log_likelihood = 0.0

        for basis in ["Z", "X"]:
            p = basis_probabilities(
                candidate_key, depth, basis
            )

            counts = observations[basis]

            log_likelihood += np.sum(
                counts * np.log(np.maximum(p, 1e-12))
            )

        scores.append((candidate_key, log_likelihood))

    scores.sort(key=lambda x: x[1], reverse=True)
    return scores

results = attack(depth=4)

print("Actual secret key:", SECRET_KEY)
print("\nTop 10 candidates:")

for key, score in results[:10]:
    print(f"Key {key:2d}: Score = {score:.3f}")

print("\nPredicted key:", results[0][0])


In [ ]:

import matplotlib.pyplot as plt

depths = [0, 1, 2, 4, 8]
trials = 30

success_rates = []

for depth in depths:
    correct = 0

    for trial in range(trials):
        scores = attack(depth)
        predicted = scores[0][0]

        if predicted == SECRET_KEY:
            correct += 1

    rate = correct / trials
    success_rates.append(rate)

    print(
        f"Depth {depth}: "
        f"{correct}/{trials} correct "
        f"({rate:.1%})"
    )

plt.figure(figsize=(8, 5))
plt.plot(depths, success_rates, marker="o")
plt.xlabel("Scrambling circuit depth")
plt.ylabel("Key recovery success rate")
plt.title("Restricted Attack Success vs Circuit Depth")
plt.ylim(0, 1.05)
plt.grid(True)
plt.show()


In [ ]:

from qiskit.quantum_info import Statevector, state_fidelity

key = 13
depth = 4

# Create the original keyed state
original = scrambling_circuit(key, depth=0)

# Create the scrambled state
scrambled = scrambling_circuit(key, depth=depth)

# Extract only the public scrambling operation
public_scrambler = QuantumCircuit(N_QUBITS)

for layer in range(depth):
    for q in range(N_QUBITS):
        public_scrambler.ry(
            0.37 * (q + 1) * (layer + 1), q
        )
        public_scrambler.rz(
            0.23 * (q + 1) * (layer + 1), q
        )

    public_scrambler.cx(0, 1)
    public_scrambler.cx(2, 3)
    public_scrambler.cx(1, 2)
    public_scrambler.cx(3, 0)

# Undo the public scrambling
recovery = scrambled.compose(
    public_scrambler.inverse()
)

original_state = Statevector.from_instruction(original)
recovered_state = Statevector.from_instruction(recovery)

fidelity = state_fidelity(
    original_state,
    recovered_state
)

print("Recovery fidelity:", fidelity)


In [ ]:

# QUANTUM GRAVITY DEFENSE
# Experiment 7: Key-Dependent Quantum Scrambling
# Cell 1: Environment setup

%pip -q install "qiskit>=2.0,<3.0" numpy matplotlib

import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, state_fidelity

# Experiment configuration
NUM_QUBITS = 4
KEY_BITS = 6
NUM_KEYS = 2 ** KEY_BITS

SECRET_KEY = 13
RANDOM_SEED = 42

rng = np.random.default_rng(RANDOM_SEED)

print("QUANTUM GRAVITY DEFENSE")
print("Experiment 7: Key-Dependent Scrambling")
print("-" * 45)
print(f"Qubits: {NUM_QUBITS}")
print(f"Key size: {KEY_BITS} bits")
print(f"Possible keys: {NUM_KEYS}")
print(f"Test secret key: {SECRET_KEY}")
print("Qiskit environment initialized successfully!")


In [ ]:

# CELL 2: Key-dependent quantum state generator

def key_to_bits(key, width=KEY_BITS):
    """Convert an integer key into a fixed-length bit string."""
    if not 0 <= key < 2**width:
        raise ValueError("Key is outside the allowed range.")
    return format(key, f"0{width}b")


def build_keyed_circuit(key, depth=2):
    """Build a keyed 4-qubit quantum circuit."""
    bits = key_to_bits(key)
    qc = QuantumCircuit(NUM_QUBITS)

    # STAGE 1: Encode the key
    for q in range(NUM_QUBITS):
        angle = np.pi / 4 + int(bits[q]) * np.pi / 2
        qc.ry(angle, q)

    qc.barrier()

    # STAGE 2: Key-dependent scrambling
    for layer in range(depth):

        # Key-dependent rotations
        for q in range(NUM_QUBITS):
            bit_index = (q + layer) % KEY_BITS
            bit = int(bits[bit_index])

            theta = np.pi / 8 + bit * np.pi / 3
            qc.rz(theta, q)

            phi = np.pi / 7 + bit * np.pi / 5
            qc.ry(phi, q)

        # Entangling gates
        for q in range(NUM_QUBITS - 1):
            qc.cx(q, q + 1)

        qc.barrier()

    return qc


# Build the circuit for our test key
circuit = build_keyed_circuit(SECRET_KEY, depth=2)

print("Secret key:", SECRET_KEY)
print("Binary key:", key_to_bits(SECRET_KEY))
print()
print(circuit.draw(output="text"))
